In [5]:
import pandas as pd
from pathlib import Path
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier

project = Path.cwd().parent

features = pd.read_csv(
    project / "data" / "processed" / "customer_features.csv"
)

features["SIGNUP_DATE"] = pd.to_datetime(features["SIGNUP_DATE"])
features = features.sort_values("SIGNUP_DATE").reset_index(drop=True)

cutoff_date = features["SIGNUP_DATE"].quantile(0.8)

train_mask = features["SIGNUP_DATE"] <= cutoff_date
test_mask = features["SIGNUP_DATE"] > cutoff_date

X = features.drop(columns=["CHURNED", "CUSTOMER_ID", "SIGNUP_DATE"])
y = features["CHURNED"].astype(int)

X_train = X.loc[train_mask].copy()
X_test = X.loc[test_mask].copy()
y_train = y.loc[train_mask].copy()
y_test = y.loc[test_mask].copy()

categorical_columns = X_train.select_dtypes(
    include=["object", "string"]
).columns.tolist()

numeric_columns = X_train.select_dtypes(
    exclude=["object", "string"]
).columns.tolist()

preprocessor = ColumnTransformer([
    (
        "num",
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]),
        numeric_columns
    ),
    (
        "cat",
        Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore"))
        ]),
        categorical_columns
    )
])

random_forest_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

random_forest_model.fit(X_train, y_train)

rf_preprocessor = random_forest_model.named_steps["preprocessor"]
rf_model = random_forest_model.named_steps["model"]

feature_names = rf_preprocessor.get_feature_names_out()

importance = pd.DataFrame({
    "Feature": feature_names,
    "Importance": rf_model.feature_importances_
})

importance = importance.sort_values(
    "Importance", ascending=False
).reset_index(drop=True)

importance.head(15)

,Feature,Importance
0,num__transaction_change,0.147967
1,num__spend_change,0.100357
2,num__recent_transactions,0.091492
3,num__app_session_change,0.086367
4,num__recent_spend,0.083681
5,num__recent_app_sessions,0.082914
6,num__recent_coupons,0.062237
7,num__recent_emails_opened,0.041125
8,num__active_months,0.040449
9,num__total_transactions,0.027334


In [6]:
import pandas as pd

analysis = features.copy()

analysis["SIGNUP_COHORT"] = pd.cut(
    analysis["SIGNUP_DATE"],
    bins=4,
    labels=["Early", "Early-Mid", "Late-Mid", "Recent"]
)

tier_risk = (
    analysis.groupby("MEMBERSHIP_TIER", observed=True)["CHURNED"]
    .agg(["count", "sum", "mean"])
    .reset_index()
    .rename(columns={
        "count": "Customers",
        "sum": "Churned",
        "mean": "Churn_Rate"
    })
)

city_risk = (
    analysis.groupby("CITY", observed=True)["CHURNED"]
    .agg(["count", "sum", "mean"])
    .reset_index()
    .rename(columns={
        "count": "Customers",
        "sum": "Churned",
        "mean": "Churn_Rate"
    })
    .sort_values("Churn_Rate", ascending=False)
)

cohort_risk = (
    analysis.groupby("SIGNUP_COHORT", observed=True)["CHURNED"]
    .agg(["count", "sum", "mean"])
    .reset_index()
    .rename(columns={
        "count": "Customers",
        "sum": "Churned",
        "mean": "Churn_Rate"
    })
)

tier_risk["Churn_Rate"] = (tier_risk["Churn_Rate"] * 100).round(2)
city_risk["Churn_Rate"] = (city_risk["Churn_Rate"] * 100).round(2)
cohort_risk["Churn_Rate"] = (cohort_risk["Churn_Rate"] * 100).round(2)

print("Tier")
display(tier_risk)

print("City")
display(city_risk)

print("Signup Cohort")
display(cohort_risk)

Tier


,MEMBERSHIP_TIER,Customers,Churned,Churn_Rate
0,Gold,697,195,27.98
1,Platinum,260,77,29.62
2,Silver,1268,390,30.76


City


,CITY,Customers,Churned,Churn_Rate
0,Austin,222,80,36.04
2,Columbus,223,76,34.08
5,Nashville,207,63,30.43
7,Raleigh,254,77,30.31
1,Boise,209,62,29.67
3,Denver,226,65,28.76
9,Tulsa,253,72,28.46
4,Madison,219,61,27.85
6,Portland,205,54,26.34
8,Spokane,207,52,25.12


Signup Cohort


,SIGNUP_COHORT,Customers,Churned,Churn_Rate
0,Early,577,202,35.01
1,Early-Mid,546,203,37.18
2,Late-Mid,555,180,32.43
3,Recent,547,77,14.08


In [8]:
import numpy as np

probabilities = random_forest_model.predict_proba(X_test)[:, 1]

prediction_output = pd.DataFrame({
    "CUSTOMER_ID": features.loc[test_mask, "CUSTOMER_ID"].values,
    "CHURN_PROBABILITY": probabilities,
    "PREDICTED_CHURN": (probabilities >= 0.70).astype(int)
})

top_features = importance.head(10)["Feature"].str.replace(
    "num__", "", regex=False
).tolist()

prediction_output["KEY_DRIVERS"] = ", ".join(top_features)

prediction_output["CHURN_PROBABILITY"] = (
    prediction_output["CHURN_PROBABILITY"].round(4)
)

prediction_output.to_csv(
    project / "outputs" / "churn_predictions.csv",
    index=False
)

prediction_output.head(10)

,CUSTOMER_ID,CHURN_PROBABILITY,PREDICTED_CHURN,KEY_DRIVERS
0,500885,0.9633,1,"transaction_change, spend_change, recent_trans..."
1,500958,0.6800,0,"transaction_change, spend_change, recent_trans..."
2,502565,0.0367,0,"transaction_change, spend_change, recent_trans..."
3,502284,0.9867,1,"transaction_change, spend_change, recent_trans..."
4,500895,0.8067,1,"transaction_change, spend_change, recent_trans..."
5,502381,0.0367,0,"transaction_change, spend_change, recent_trans..."
6,500618,0.0567,0,"transaction_change, spend_change, recent_trans..."
7,500085,0.4167,0,"transaction_change, spend_change, recent_trans..."
8,501154,0.0667,0,"transaction_change, spend_change, recent_trans..."
9,501865,0.1167,0,"transaction_change, spend_change, recent_trans..."
